In [1]:
!pip install pyspark==3.3.0

In [1]:
import time

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, udf
from pyspark.sql.types import *

In [2]:
# Schema for the Kafka JSON message
schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("product_id", StringType(), True),
            StructField("description", StringType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("product_id", StringType(), True),
            StructField("description", StringType(), True)
        ]), True),
        StructField("source", StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("ts_us", LongType(), True),
            StructField("ts_ns", LongType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", LongType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])

In [3]:
spark = SparkSession \
    .builder \
    .appName("Streaming from Kafka") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,com.datastax.spark:spark-cassandra-connector_2.12:3.3.0') \
    .config('spark.cassandra.connection.host', 'cassandra') \
    .config('spark.cassandra.connection.port', '9042') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

In [4]:
drugs_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.his_drugs") \
    .option("startingOffsets", "earliest") \
    .load()

drugs_stream_df = drugs_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
drugs_parsedData = drugs_stream_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
drugs_flattenedData = drugs_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

drugs_pipeline = drugs_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_amo") \
                  .option("table", "his_acts") \
                  .mode("append") \
                  .save()) \
    .start()

In [5]:
devices_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.his_devices") \
    .option("startingOffsets", "earliest") \
    .load()

devices_stream_df = devices_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
devices_parsedData = devices_stream_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
devices_flattenedData = devices_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

devices_pipeline = devices_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_amo") \
                  .option("table", "his_acts") \
                  .mode("append") \
                  .save()) \
    .start()

In [6]:
medicals_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.his_medicals") \
    .option("startingOffsets", "earliest") \
    .load()

medicals_stream_df = medicals_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
medicals_parsedData = medicals_stream_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
medicals_flattenedData = medicals_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

medicals_pipeline = medicals_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_amo") \
                  .option("table", "his_acts") \
                  .mode("append") \
                  .save()) \
    .start()

In [7]:
blabla_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.his_blabla") \
    .option("startingOffsets", "earliest") \
    .load()

blabla_stream_df = blabla_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
blabla_parsedData = blabla_stream_df.select(from_json(col("value"), schema).alias("data"))

# Flatten the data and select fields
blabla_flattenedData = blabla_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

blabla_pipeline = blabla_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_amo") \
                  .option("table", "his_acts") \
                  .mode("append") \
                  .save()) \
    .start()

In [8]:
from threading import Thread

In [ ]:

# List of all streaming queries
queries = [drugs_pipeline, devices_pipeline, medicals_pipeline, blabla_pipeline]

# Function to await termination for a single query
def await_query(query):
    try:
        query.awaitTermination()
    except Exception as e:
        print(f"Error in query: {e}")

# Create threads for each query
threads = [Thread(target=await_query, args=(query,)) for query in queries]

# Start all threads
for thread in threads:
    thread.start()

# Block the notebook until all threads finish (optional)
for thread in threads:
    thread.join()

Error in query: Query [id = 396b2297-b3d6-4553-b2da-b4bc85fa5eaa, runId = 3551f6a6-6521-4a3f-b43a-cfda407ef004] terminated with exception: An exception was raised by the Python Proxy. Return Message: Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/py4j/clientserver.py", line 617, in _call_proxy
    return_value = getattr(self.pool[obj_id], method)(*params)
  File "/opt/conda/lib/python3.10/site-packages/pyspark/sql/utils.py", line 272, in call
    raise e
  File "/opt/conda/lib/python3.10/site-packages/pyspark/sql/utils.py", line 269, in call
    self.func(DataFrame(jdf, self.session), batch_id)
  File "/tmp/ipykernel_511/1983183095.py", line 26, in <lambda>
    .save()) \
  File "/opt/conda/lib/python3.10/site-packages/pyspark/sql/readwriter.py", line 966, in save
    self._jwrite.save()
  File "/opt/conda/lib/python3.10/site-packages/py4j/java_gateway.py", line 1321, in __call__
    return_value = get_return_value(
  File "/opt/conda/lib/python3.10/